In [ ]:
import numpy as np
import scipy
import plotly.express as px

# The Idea

If you know nothing about the preferences of a specific animal and are asked what it's distribution might look like you're going to pick an even distribution? Why? Because it is the least informed guess you can make, or in the verbage of information theory it is the guess with the highest entropy. 

However suppose now you have a local preference model - a movement model. Well now an evenly distributed guess actually has relatively little entropy over time because a very specific distribution has to be assumed in order for your fish to end up at an even distribution at this point in time. So we clearly need a new guess. 

What should that guess be? Well it should incorporate two principles:

1. It should be static. I.e., our guess shouldn't change quickly over the passage of time as the preferences of our creature are likely not changing that quickly. 
2. It should be maximally entropic - this is secondary to the first and is what gets us away from an even distribution guess.

So let's see how this works out!

# Encoding a Preference/Movement Model

We can imagine such a model as existing over a set of states (which could be grid points as an example). Then we can encode the movement likelihoods in a matrix $M$ where if $x$ are the current states, $Mx$ are the expected upcoming states. That is the rows of $M$ indicate the state moved to and columns the state moved from. Cleary the sum along the columns has to be one. 

Let's start with a very simple preference model where we have a series of states along a line. Each of those will have a feature that's either 10 or 1 indicating a kind of "preference".

In [ ]:
states = [1] * 10 + [10, 10, 10] + [1] * 10 + [10, 10, 10]

Next we can model the likelihood of movement from one of these cells to another as the state's value divided by the sum value. So if you're surrounded by neighors that all have the same value our hypothetical creatures will move around (or stay) at random. But if there's a spot that has a high value compared to the average, individuals will be more likely to go there. 

In [ ]:
M = np.zeros((len(states), len(states)))
for i in range(len(states)):
    neighbors = [i - 1, i, i + 1]
    neighbors = [j for j in neighbors if j >= 0 and j < len(states)]
    vals = [states[j] for j in neighbors]
    for k, j in enumerate(neighbors):
        M[j, i] = vals[k] / np.sum(vals)

# Setting up the Optimization

First our optimization is over the probabilities we assign to each state in our guess. So let's initialize that to the even guess we know is going to be wrong (so we can see the optimization do something).

In [ ]:
guess = np.ones(M.shape[0])
guess = guess / guess.sum()

Next we need to make sure in the process of optimizing that our guess's probabilities sum to one and that the individual probabilities are between 0 and 1.

In [ ]:
constraints = []
bounds = bounds = [
    (0, 1) for _ in guess
]
constraints.append(
    scipy.optimize.LinearConstraint(
        np.ones(guess.shape[0]), lb=1, ub=1
    ),
)

Then we want to make sure our guess is relatively static under movment. That is:

$$-\delta x \leq (M-I)x \leq \delta x$$

In [ ]:
delta = 0.01
constraints.extend([
    scipy.optimize.LinearConstraint(
        M - np.identity(guess.shape[0]) * (1 + delta), lb=-float("inf"), ub=0
    ),
    scipy.optimize.LinearConstraint(
        M - np.identity(guess.shape[0]) * (1 - delta), lb=0, ub=float("inf")
    )
])

Finally `scipy` is going to want to minimize something so we'll minimize the negative entropy (and thereby maximize the entropy).

In [ ]:
def negative_entropy(p):
    return (p[p > 0] * np.log(p[p > 0])).sum()

In [ ]:
result = scipy.optimize.minimize(
    negative_entropy, guess, 
    bounds=bounds, 
    constraints=constraints
)
result

We can then check to see what the maximum % delta actually was

In [ ]:
x = result.x 
np.max(np.abs((x - np.matmul(M, x)) / x))

And then plot our guess!

In [ ]:
px.bar(x)

# Other Examples

In [ ]:
def guestimate(M, delta):
    guess = np.ones(M.shape[0])
    guess = guess / guess.sum()

    constraints = []
    bounds = bounds = [
        (0, 1) for _ in guess
    ]
    constraints.append(
        scipy.optimize.LinearConstraint(
            np.ones(guess.shape[0]), lb=1, ub=1
        ),
    )

    constraints.extend([
        scipy.optimize.LinearConstraint(
            M - np.identity(guess.shape[0]) * (1 + delta), lb=-float("inf"), ub=0
        ),
        scipy.optimize.LinearConstraint(
            M - np.identity(guess.shape[0]) * (1 - delta), lb=0, ub=float("inf")
        )
    ])

    return scipy.optimize.minimize(
        negative_entropy, guess, 
        bounds=bounds, 
        constraints=constraints
    )

## An Island

In [ ]:
X, Y = np.meshgrid(np.arange(-1, 1, 0.1), np.arange(-1, 1, 0.1))
X = X.flatten()
Y = Y.flatten()
num_states = X.shape[0]

U = np.ones(num_states)
U[np.sqrt(X ** 2 + Y ** 2) < 0.3] = 10


px.scatter(x=X, y=Y, color=U, height=600, width=600)

In [ ]:
max_distance = 0.2
M = np.zeros([num_states, num_states])
for i in range(num_states):
    neighbors = [j for j in range(num_states) if np.sqrt((X[i] - X[j]) ** 2 + (Y[i] - Y[j]) ** 2) <= max_distance]
    total = sum([U[j] for j in neighbors])
    for j in neighbors:
        M[j, i] = U[j] / total

In [ ]:
result = guestimate(M, 0.01)

In [ ]:
x = result.x 
np.max(np.abs((x - np.matmul(M, x)) / x))

In [ ]:
px.scatter(x=X, y=Y, color=result.x, height=600, width=600)

In [ ]:
_filter = np.abs(Y) < 10 ** -10
px.bar(x=X[_filter], y=result.x[_filter])

## A Donut

In [ ]:
X, Y = np.meshgrid(np.arange(-1, 1, 0.1), np.arange(-1, 1, 0.1))
X = X.flatten()
Y = Y.flatten()
num_states = X.shape[0]

U = np.ones(num_states)
U[(np.sqrt(X ** 2 + Y ** 2) > 0.4) & (np.sqrt(X ** 2 + Y ** 2) < 0.7)] = 10


px.scatter(x=X, y=Y, color=U, height=600, width=600)

In [ ]:
max_distance = 0.2
M = np.zeros([num_states, num_states])
for i in range(num_states):
    neighbors = [j for j in range(num_states) if np.sqrt((X[i] - X[j]) ** 2 + (Y[i] - Y[j]) ** 2) <= max_distance]
    total = sum([U[j] for j in neighbors])
    for j in neighbors:
        M[j, i] = U[j] / total

In [ ]:
result = guestimate(M, 0.01)

In [ ]:
x = result.x 
np.max(np.abs((x - np.matmul(M, x)) / x))

In [ ]:
px.scatter(x=X, y=Y, color=result.x, height=600, width=600)

In [ ]:
_filter = np.abs(Y) < 10 ** -10
px.bar(x=X[_filter], y=result.x[_filter])

## A Merry Go Round

In [ ]:
M = np.array([
    [0, 1, 0],
    [0, 0, 1],
    [1, 0, 0]
])
result = guestimate(M, 0.01)

In [ ]:
result.x

## An Island on an Island

In [ ]:
X, Y = np.meshgrid(np.arange(-1, 1, 0.1), np.arange(-1, 1, 0.1))
X = X.flatten()
Y = Y.flatten()
num_states = X.shape[0]

U = np.ones(num_states)
U[np.sqrt(X ** 2 + Y ** 2) < 0.7] = 10
U[np.sqrt(X ** 2 + Y ** 2) < 0.3] = 20

px.scatter(x=X, y=Y, color=U, height=600, width=600)

In [ ]:
max_distance = 0.2
M = np.zeros([num_states, num_states])
for i in range(num_states):
    neighbors = [j for j in range(num_states) if np.sqrt((X[i] - X[j]) ** 2 + (Y[i] - Y[j]) ** 2) <= max_distance]
    total = sum([U[j] for j in neighbors])
    for j in neighbors:
        M[j, i] = U[j] / total

In [ ]:
result = guestimate(M, 0.01)

In [ ]:
x = result.x 
np.max(np.abs((x - np.matmul(M, x)) / x))

In [ ]:
px.scatter(x=X, y=Y, color=result.x, height=600, width=600)

In [ ]:
_filter = np.abs(Y) < 10 ** -10
px.bar(x=X[_filter], y=result.x[_filter])

# Now for Something Real

In [ ]:
import os
import h3
import pandas as pd
from shapely.geometry import Polygon, Point
from tqdm import tqdm

os.environ['HAVEN_DATABASE'] = 'haven'
os.environ['AWS_PROFILE'] = 'admin'

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import plot_h3_slider

In [ ]:
poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)

In [ ]:
dates = [
    '2022-01-15 12:00:00', '2022-02-15 12:00:00', '2022-03-15 12:00:00',
    '2022-04-15 12:00:00', '2022-05-15 12:00:00', '2022-06-15 12:00:00',
    '2022-07-15 12:00:00', '2022-08-15 12:00:00', '2022-09-15 12:00:00',
    '2022-10-15 12:00:00', '2022-11-15 12:00:00', '2022-12-15 12:00:00',

    '2022-01-01 12:00:00', '2022-02-01 12:00:00', '2022-03-01 12:00:00',
    '2022-04-01 12:00:00', '2022-05-01 12:00:00', '2022-06-01 12:00:00',
    '2022-07-01 12:00:00', '2022-08-01 12:00:00', '2022-09-01 12:00:00',
    '2022-10-01 12:00:00', '2022-11-01 12:00:00', '2022-12-01 12:00:00'
]

dataframes = []

for date in tqdm(dates):
    sql = f'''
    select 
        origin_h3_index,
        next_h3_index,
        time,
        probability,
        stay_put
    from 
        movement_model_full_inference_10_1_10 mm
        inner join mean_elevation_by_h3 el
            on mm.origin_h3_index = el.h3_index
            and el.h3_resolution = 4
            and el.elevation > -600
    where 
        time = TIMESTAMP '{date}'
    '''
    data = read_data_w_cache(sql)
    data['lat'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
    data['lon'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])

    data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
    data = data[data['inside_polygon'] & (data['lon'] < -145)]

    origins = sorted(data['origin_h3_index'].unique())
    data = data[data['next_h3_index'].isin(origins)]
    data['probability'] = data['probability'].fillna(0) + 0.00001
    data['total_probability'] = data.groupby('origin_h3_index')['probability'].transform('sum')
    data['probability'] = data['probability'] / data['total_probability']
    data['date'] = date
    dataframes.append(data)

full_data = pd.concat(dataframes).reset_index()
print(full_data.shape)
full_data.head()

In [ ]:
def generate_a_guestimate(data, date, delta):
    data = data[data['date'] == date]
    indices = {
        h3_index: i 
        for i, h3_index in enumerate(origins)
    }
    reverse_index = {
        i: h3_index for h3_index, i in indices.items()
    }
    num_options = data.groupby('origin_h3_index').size().to_dict()
    M = np.zeros((len(indices), len(indices)))
    N = np.zeros((len(indices), len(indices)))

    for _, row in tqdm(data.iterrows()):
        num_neighors = num_options[row['origin_h3_index']]
        i = indices[row['next_h3_index']]
        j = indices[row['origin_h3_index']]
        M[i, j] = row['probability']
        M[i, j] = row['probability']
        N[i, j] = 1 / num_neighors
    
    result = guestimate(M, delta)

    rows = []
    for i, p in enumerate(result.x):
        lat_sum = 0
        lon_sum = 0
        weight_sum = 0
        for j in range(M.shape[0]):
            weight = M[j, i]
            if weight == 0: continue
            lat, lon = h3.h3_to_geo(reverse_index[j])
            lat_sum += lat * weight
            lon_sum += lon * weight
            weight_sum += weight
        lat_end = lat_sum / weight_sum
        lon_end = lon_sum / weight_sum
        lat_start, lon_start = h3.h3_to_geo(reverse_index[i])
        rows.append({
            'p': p,
            'stickiness': M[i, i], 
            'h3_index': reverse_index[i],
            'lat_start': lat_start,
            'lon_start': lon_start,
            'lat_end': lat_end,
            'lon_end': lon_end,
            'date': date
        })
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat(
    [
        generate_a_guestimate(full_data, date, 0.01)
        for date in dates
    ]
)
results['log_p'] = np.log(results['p'])/np.log(2)
results.head()

In [ ]:
from mirrorverse.plotting import * 
from plotly.colors import sample_colorscale

def build_geojson(dataframe, id_field, geometry_field='geometry'):
    geoms = gpd.GeoDataFrame(
        dataframe[[id_field, geometry_field]]
        .rename(columns={geometry_field: 'geometry'})
    )
    geoms = json.loads(geoms.to_json())
    for feature in geoms['features']:
        feature['id'] = feature['properties'][id_field]
    return geoms 

slider_field = 'date'
dataframe = results.copy()
id_field = 'h3_index'
value_field = 'log_p'
zmin = results['log_p'].quantile(0.1)
zmax = results['log_p'].quantile(0.9)
colorscale = 'Reds'
zoom=2
center={"lat": 60, "lon": -180}

def plot_map(id_field, slider_field, dataframes, value_fields, line_value_fields={}, colorscales={}, bounds={}, zoom=2, center={"lat": 60, "lon": -180}, height=1000, width=750):
    assert len(value_fields) == len(set(value_fields)), 'Value field must be unique!'

    fig = go.Figure()
    slider_vals = sorted(dataframes[0][slider_field].unique())
    for slider_val in slider_vals:
        for dataframe, value_field in zip(dataframes, value_fields):
            zmin, zmax = bounds.get(
                value_field, (dataframe[value_field].min(), dataframe[value_field].max())
            )
            colorscale = colorscales.get(value_field, 'Reds')
            sub = dataframe[dataframe[slider_field] == slider_val]
            geojson = build_geojson(sub, id_field)

            if value_field in line_value_fields:
                line_value_field = line_value_fields[value_field]
                zmin_line, zmax_line = bounds.get(
                    line_value_field,
                    (sub[line_value_field].quantile(0.1), sub[line_value_field].quantile(0.9))
                )
                norm = np.clip((sub[line_value_field] - zmin_line) / (zmax_line - zmin_line), 0, 1)
                line_colors = sample_colorscale(colorscales.get(line_value_field, 'Reds'), norm)
                line_widths = [3 for _ in line_colors]
            else:
                line_colors = ['black' for _ in range(sub.shape[0])]
                line_widths = [1 for _ in line_colors]

            fig.add_trace(
                go.Choroplethmapbox(
                    geojson=geojson,
                    locations=sub[id_field],
                    z=sub[value_field],
                    zmin=zmin,
                    zmax=zmax,
                    colorscale=colorscale,
                    visible=False,
                    marker_opacity=0.5,
                    marker_line=dict(
                        color=line_colors,
                        width=line_widths
                    )
                )
            )

    traces_per_view = len(dataframes)
    for i in range(traces_per_view):
        fig.data[i].visible = True

    steps = []
    for i, slider_val in enumerate(slider_vals):
        step = dict(
            method="update",
            args=[
                {"visible": [False] * len(slider_vals) * traces_per_view},
                {"title": f"{slider_field}: {slider_val}"},
            ],
            label=f"{slider_val}"
        )
        for j in range(traces_per_view):
            step["args"][0]["visible"][i*traces_per_view + j] = True
        steps.append(step)
    
    sliders = [dict(
        active=0,
        currentvalue={"prefix": f"{slider_field}: "},
        pad={"t": 50, "b": 25, "l": 25},
        steps=steps
    )]

    fig.update_layout(
        sliders=sliders
    )

    fig.update_layout(
        margin={"r":0,"t":30,"l":0,"b":0}, mapbox=dict(style="carto-positron", zoom=zoom, center = center),
        height=height, width=width
    )

    return fig
    
add_h3_geoms(results)
lines = results.copy()
lines['geometry'] = lines.apply(lambda r: Polygon([(r['lon_start'], r['lat_start']), (r['lon_end'], r['lat_end']), (r['lon_end'] + 0.05, r['lat_end']), (r['lon_start'] + 0.05, r['lat_start'])]), axis=1)

dataframes = [results, lines]
id_field = 'h3_index'
slider_field = 'date'
value_fields = ['log_p', 'stickiness']
line_value_fields = {
    'log_p': 'stickiness'
}
bounds = {
    'log_p': (results['log_p'].quantile(0.1), results['log_p'].quantile(0.9)),
}
colorscales = {
    'stickiness': 'Blues'
}
plot_map(id_field, slider_field, dataframes, value_fields, line_value_fields=line_value_fields, colorscales=colorscales, bounds=bounds).show()